## Planning:
- Load data
- Create sample FE - sum of boolean and proportion of boolean and Digit Position
- Create TE in run.cv
- Implement for XGBoost

In [1]:
import pandas as pd
import numpy as np

In [2]:
from pathlib import Path

In [3]:
current_dir = Path.cwd()
competition_root = current_dir.parents[1]

In [4]:
competition_path = competition_root/"Predicting Credit Card Fraud"

In [42]:
train = pd.read_csv(competition_path/"data/train.csv")
test = pd.read_csv(competition_path/"data/test.csv")
submit = pd.read_csv(competition_path/"data/sample_submission.csv")

In [43]:
# check correlation of numeric variable
# take all column numeric column
numeric_cols = train.drop(columns='id').select_dtypes(include='number').columns.tolist()
numeric_cols_ex = numeric_cols.copy()
numeric_cols_ex.remove("is_fraud")

In [44]:
bool_cols = train.select_dtypes(include='bool').columns.tolist()

In [98]:
obj_cols = train.select_dtypes(include='object').columns.tolist()

In [46]:
# checking the distribution on each of category in categorical variable
for col in obj_cols:
    counts = train.groupby(col)["id"].count()
    proportion = counts/counts.sum()

    print(f'col: {col}')
    summary = pd.DataFrame({'count': counts, 'proportion': proportion})
    print(summary)
    print('-'*20)

col: merchant_category
                    count  proportion
merchant_category                    
Crypto Exchange      5285    0.007550
Electronics         48693    0.069565
Fuel                59880    0.085548
Gaming              23784    0.033979
Gift Cards           9456    0.013509
Groceries          142578    0.203694
Healthcare          35889    0.051273
Online Retail      155097    0.221580
Restaurants        101360    0.144808
Streaming           47196    0.067427
Travel              28715    0.041024
Utilities           42027    0.060042
--------------------
col: card_type
             count  proportion
card_type                     
Amex         45797    0.065428
Discover     17235    0.024623
Mastercard  254857    0.364102
RuPay        37070    0.052960
Visa        345001    0.492887
--------------------
col: auth_method
                    count  proportion
auth_method                          
3D Secure          210036    0.300069
Biometric           93666    0.133816
No

## Feature Engineering - 1st Digit and 2 first Digit - Benford's Law

In [74]:
def extract_first_digit_features(df, numeric_cols,n=1):
    """Extracts the leading digits (e.g., 1st digit or first 2 digits)

    from numeric columns for Benford's Law analysis.

    Parameters:
    - df: pandas DataFrame
    - numeric_cols: list of columns to process
    - n_digits: number of leading digits to extract (default is 1)
    """
    df = df.copy()

    for col in numeric_cols:
        # 1. Convert to string, strip all non-digit characters (decimals, signs)
        str_series = (
            df[col].astype(str).str.replace(r"\D+", "", regex=True).str.lstrip("0")
        )

        # 2. Extract the first n digits (handling empty strings where values were 0)
        feature_name = (
            f"{col}_first_digit"
        )

        # 3. Extract the first n characters, handle blanks as 'Missing' or NaN,
        # and convert the result into a categorical type
        extracted = (
            str_series.str[:n].replace("", np.nan).fillna("Missing")
        )  # Using string 'Missing' to keep categories clean

        # Extract slice and handle cases where string length is less than n_digits
        df[feature_name] = extracted.astype("category")

    return df

In [75]:
def extract_first_two_digits(df, numeric_cols):
  """Extracts the first two leading digits from numeric columns.

  Handles negatives, decimals, and leading zeros (e.g., 0.0034 -> 34).

  Parameters:
  - df: pandas DataFrame
  - numeric_cols: list of column names to process

  Returns:
  - DataFrame with new columns named '{col}_first_two_digits'
  """
  df = df.copy()

  for col in numeric_cols:
    # 1. Convert to string and remove non-digit characters (signs, decimals, exponents)
    # Then strip any leading zeros
    str_series = (
        df[col].astype(str).str.replace(r"[^\d]", "", regex=True).str.lstrip("0")
    )

    # 2. Extract the first 2 characters, handling empty strings (e.g., original value was 0)
    extracted = str_series.str[:2]

    # 3. Format into a clean numeric feature (using -1 for missing/zero values)
    feature_name = f"{col}_first_two_digits"
    df[feature_name] = (
        extracted.replace("", np.nan).fillna("Missing").astype("category")
    )

  return df

In [76]:
numeric_features = ["amount_usd", "account_balance_usd"]

In [80]:
train_featured = extract_first_digit_features(train, numeric_features)

In [81]:
train_featured = extract_first_two_digits(train_featured, numeric_features)

In [82]:
test_featured = extract_first_digit_features(test, numeric_features)

In [83]:
test_featured = extract_first_two_digits(test_featured, numeric_features)

In [134]:
# train_featured[numeric_features+["extract_first_digit_features","extract_first_two_digits"]]
train_featured.shape

(699960, 30)

In [133]:
test_featured.shape

(299984, 29)

In [97]:
cat_cols = train_featured.select_dtypes(include='category').columns.tolist()

In [93]:
cat_cols

['amount_usd_first_digit',
 'account_balance_usd_first_digit',
 'amount_usd_first_two_digits',
 'account_balance_usd_first_two_digits']

In [99]:
category_cols = obj_cols + cat_cols

In [135]:
category_cols

['merchant_category',
 'card_type',
 'auth_method',
 'channel',
 'device_type',
 'amount_usd_first_digit',
 'account_balance_usd_first_digit',
 'amount_usd_first_two_digits',
 'account_balance_usd_first_two_digits']

In [102]:
# cross tab cat_cols with output
def analyze_categorical_fraud(df, cat_cols, target_col="is_fraud"):
  """Generates cross-tabulations (counts and percentages) between

  categorical columns and the target (is_fraud) column.

  Parameters:
  - df: pandas DataFrame
  - cat_cols: list of categorical column names
  - target_col: name of the target column (default 'is_fraud')
  """
  fraud_analysis = {}

  for col in cat_cols:
    print(f"========================================")
    print(f"Cross-Tabulation for: {col} vs {target_col}")
    print(f"========================================")

    # 1. Absolute counts cross-tabulation (with margins/totals)
    count_tab = pd.crosstab(df[col], df[target_col], margins=True)

    # 2. Row percentages (shows % of fraud vs non-fraud within each category)
    pct_tab = pd.crosstab(
        df[col], df[target_col], normalize="index", margins=True
    ) * 100

    # Combine or print them clearly
    print("\n--- Counts ---")
    print(count_tab)

    print("\n--- Row Percentages (%) ---")
    print(pct_tab.round(2))
    print("\n")

    # Store in a dictionary if you want to inspect them later programmatically
    fraud_analysis[col] = {"counts": count_tab, "percentages": pct_tab}

  return fraud_analysis

In [105]:
fr_analysis = analyze_categorical_fraud(train_featured, category_cols, target_col="is_fraud")

Cross-Tabulation for: amount_usd_first_digit vs is_fraud

--- Counts ---
is_fraud                   0.0    1.0     All
amount_usd_first_digit                       
1                       173327   2968  176295
2                       147508   2236  149744
3                       116772   1597  118369
4                        92776   1962   94738
5                        64507   1183   65690
6                        30418    540   30958
7                        25550    510   26060
8                        18453    349   18802
9                        18960    344   19304
All                     688271  11689  699960

--- Row Percentages (%) ---
is_fraud                  0.0   1.0
amount_usd_first_digit             
1                       98.32  1.68
2                       98.51  1.49
3                       98.65  1.35
4                       97.93  2.07
5                       98.20  1.80
6                       98.26  1.74
7                       98.04  1.96
8                     

In [108]:
# fr_analysis

## Model using XGboost + TE + FE Digit

In [109]:
from sklearn.preprocessing import TargetEncoder

In [110]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

In [111]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

In [112]:
import xgboost as xgb

In [113]:
X = train_featured.drop(columns=["id","is_fraud"]).copy()
y = train_featured["is_fraud"].copy()

### Insert TE in the run_cv loop

In [128]:
def run_cv(fit_predict_fn, te_cols, name): # adding input te_cols for categorical variable required for TE
    oof = np.zeros(len(X))
    test_pred = np.zeros(len(test))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(FOLDS):
        # splitting into training and evaluation
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr, y_va = y[tr_idx].copy(), y[va_idx].copy()
        # --- FIX: Fill NaNs and ensure string type for TE columns ---
        #for col in te_cols:
        #    X_tr.loc[:,col] = X_tr.loc[:,col].fillna("Missing").astype(str)
        #   X_va.loc[:,col] = X_va.loc[:,col].fillna("Missing").astype(str)
        #   X_test.loc[:,col] = X_test.loc[:,col].fillna("Missing").astype(str)
        
        # -----------------------------------------------------------
        # creating TE model for training data
        te = TargetEncoder(smooth="auto",target_type='binary')
        
        # fit on tr data
        te.fit(X_tr[te_cols],y_tr)

        # transform on tr data
        X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])

        # transform on va data
        X_va.loc[:, te_cols] = te.transform(X_va[te_cols])
        
        # transform on X_test data
        X_test = test_featured.drop(columns=["id"]).copy()
        # X_test_obj = X_test[obj_cols].copy()
        print(X_test.info())
        X_test.loc[:, te_cols] = te.transform(X_test[obj_cols])

        # print(f'x_test',X_test.head(2))

        # print(f'X_tr_col: ',X_tr.info())
        for col in te_cols:
            X_tr[col] = X_tr[col].astype('float64')
            X_va[col] = X_va[col].astype('float64')
            X_test[col] = X_test[col].astype('float64')
            # X_tr.loc[:,col] = X_tr.loc[:,col].astype('float64')
            # X_va.loc[:,col] = X_va.loc[:,col].astype('float64')
            # X_test.loc[:,col] = X_test.loc[:,col].astype('float64')

        # print(f'x_tr ', X_tr.info())
        # print(f'x_va ', X_va.info())
        # print(f'x_test ', X_test.info())
        # start predit
        va_pred, te_pred = fit_predict_fn(X_tr, y_tr, X_va, y_va,X_test)

        # save prediction in oof and in test
        oof[va_idx] = va_pred
        test_pred += te_pred / N_FOLDS
        auc = roc_auc_score(y_va, va_pred)
        scores.append(auc)
        print(f'  fold {fold}: AUC = {auc:.5f}')
    total = roc_auc_score(y, oof)
    print(f'{name} | OOF AUC = {total:.5f} (mean {np.mean(scores):.5f} ± {np.std(scores):.5f})')
    return oof, test_pred, total

In [120]:
def fit_xgb(X_tr, y_tr, X_va, y_va,X_test):
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        n_estimators=10000,
        learning_rate=0.03,
        max_depth=7,
        min_child_weight=20,
        colsample_bytree=0.8,
        subsample=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        tree_method='hist',
        enable_categorical=True,
        early_stopping_rounds=200,
        random_state=SEED,
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    return (model.predict_proba(X_va)[:, 1],
            model.predict_proba(X_test)[:, 1])

In [121]:
def save_sub(pred_1,pred_2, name_1,name_2):
    sub = submit.copy()
    sub['is_fraud'] = pred_1
    sub.to_csv(name_1, index=False)
    print('save:', name_1)

    oof_df = train[['id']].copy()
    oof_df.loc[:,name_2] = oof_xgb
    oof_df.to_csv(name_2,index=False)
    print('save:', name_2)

In [122]:
SEED = 2312
N_FOLDS = 5
TARGET = 'is_fraud'

# Stratified KFold cross validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))

In [129]:
oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, cat_cols, 'XGBoost')

C:\Users\User\AppData\Local\Temp\ipykernel_15912\1072817443.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01705231 0.01489253 0.01705231 ... 0.01361512 0.01691813 0.01361512]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_15912\1072817443.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01686482 0.01721842 0.01686482 ... 0.01572138 0.01686482 0.01572138]' has dtype incompatible with category, please explicitly cast to a compatible dtype first.
  X_tr.loc[:, te_cols] = te.transform(X_tr[te_cols])
C:\Users\User\AppData\Local\Temp\ipykernel_15912\1072817443.py:23: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[0.01710454 0

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 299984 entries, 0 to 299983
Data columns (total 28 columns):
 #   Column                                Non-Null Count   Dtype   
---  ------                                --------------   -----   
 0   amount_usd                            299984 non-null  float64 
 1   hours_since_last_txn                  299984 non-null  float64 
 2   txn_count_last_24h                    299984 non-null  int64   
 3   distance_from_home_km                 299984 non-null  float64 
 4   card_age_months                       299984 non-null  int64   
 5   customer_age                          299984 non-null  int64   
 6   account_balance_usd                   299984 non-null  float64 
 7   cvv_retry_count                       299984 non-null  int64   
 8   velocity_score                        299984 non-null  float64 
 9   time_of_day_hour                      299984 non-null  int64   
 10  day_of_week                           299984 non-null  i

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- auth_method
- card_type
- channel
- device_type
- merchant_category
Feature names seen at fit time, yet now missing:
- account_balance_usd_first_digit
- account_balance_usd_first_two_digits
- amount_usd_first_digit
- amount_usd_first_two_digits


In [25]:
save_sub(test_xgb, oof_xgb, f'sub/sub_xgb_te_digitv2_{SEED}.csv', f'oof/oof_xgb_te_digitv2_{SEED}.csv')

save: sub/sub_xgb_te_digitv2_2312.csv
save: oof/oof_xgb_te_digitv2_2312.csv


In [25]:
seed = [42,67,1508,2312,106]

In [26]:
auc_xgb_dict = {}

In [28]:
for Seed in seed:
    SEED = Seed
    N_FOLDS = 5
    TARGET = 'is_fraud'

    # Stratified KFold cross validation
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    FOLDS = list(skf.split(X, y))

    oof_xgb, test_xgb, auc_xgb = run_cv(fit_xgb, obj_cols, 'XGBoost')
    save_sub(test_xgb, oof_xgb, f'sub/sub_xgb_te_{SEED}.csv', f'oof/oof_xgb_te_{SEED}.csv')
    auc_xgb_dict[Seed] = auc_xgb

  fold 0: AUC = 0.94455
  fold 1: AUC = 0.94846
  fold 2: AUC = 0.95111
  fold 3: AUC = 0.94614
  fold 4: AUC = 0.94221
XGBoost | OOF AUC = 0.94646 (mean 0.94649 ± 0.00308)
save: sub/sub_xgb_te_42.csv
save: oof/oof_xgb_te_42.csv
  fold 0: AUC = 0.94719
  fold 1: AUC = 0.94695
  fold 2: AUC = 0.94497
  fold 3: AUC = 0.94703
  fold 4: AUC = 0.94649
XGBoost | OOF AUC = 0.94650 (mean 0.94653 ± 0.00081)
save: sub/sub_xgb_te_67.csv
save: oof/oof_xgb_te_67.csv
  fold 0: AUC = 0.94478
  fold 1: AUC = 0.94771
  fold 2: AUC = 0.94872
  fold 3: AUC = 0.94711
  fold 4: AUC = 0.94394
XGBoost | OOF AUC = 0.94642 (mean 0.94645 ± 0.00180)
save: sub/sub_xgb_te_1508.csv
save: oof/oof_xgb_te_1508.csv
  fold 0: AUC = 0.95012
  fold 1: AUC = 0.95131
  fold 2: AUC = 0.94461
  fold 3: AUC = 0.94382
  fold 4: AUC = 0.94205
XGBoost | OOF AUC = 0.94630 (mean 0.94638 ± 0.00365)
save: sub/sub_xgb_te_2312.csv
save: oof/oof_xgb_te_2312.csv
  fold 0: AUC = 0.94696
  fold 1: AUC = 0.94823
  fold 2: AUC = 0.94669
  fo